# 25 - Modelos propios para mejorar H3-H5 y reemplazar al ingeniero

El objetivo de este cuaderno es construir pronosticos que no necesiten el estimado del ingeniero. El ingeniero aparece solo al final como punto de comparacion.

La pista de los cuadernos anteriores es que MINICARNATION y CARNATION concentran gran parte de la brecha en H3-H5. Por eso probaremos una estrategia de modelos propios: conservar la combinacion semanal que ya funciono en H1-H2 y, en H3-H5, entrenar modelos especializados por producto para MINICARNATION y CARNATION. Para los demas productos se mantiene el modelo separado por horizonte.

El entrenamiento usa produccion historica, teoria, calendario y tendencias disponibles al origen. Los hiperparametros por horizonte se toman del tuning previo del cuaderno 24, hecho en noviembre; las semanas recientes no se usan para escogerlos. En cada origen, solo entran al entrenamiento semanas reales ya cerradas.

Este es un experimento acotado: permite saber si la especializacion aporta valor; no afirma que el resultado ya sea estable.


## 1. Cargar los casos y asegurar una evaluacion comparable

Se reutiliza la misma base analitica, el mismo cruce estricto con el estimado del ingeniero y la misma ventana reciente del cuaderno 24. El ingeniero ayuda a definir la poblacion comparable y luego sirve para medir; no se usa para entrenar ni para generar el pronostico candidato.


In [1]:
# Datos comunes y evaluación sin fuga temporal
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Datos_analiticos_proyecto").is_dir())
DATA = ROOT / "Datos_analiticos_proyecto"
OUT = DATA / "experimentos_actualizacion_20261020"
OUT.mkdir(parents=True, exist_ok=True)
DEST = DATA / "modelo_con_proyecciones_teoricas"
K = ["finca", "producto", "color", "variedad"]
CASE = K + ["origen", "semana_objetivo", "horizonte"]

# Features calculadas al origen. y se sustituye por la producción 2026 recién limpia,
# pero las entradas históricas del caso quedan congeladas como estaban al emitir.
base = pd.read_parquet(DEST / "base_analitica_proyecciones.parquet")
real26 = pd.read_parquet(DATA / "produccion_real_2026_actualizada.parquet")
real26 = real26.loc[real26.finca.eq("GAITANA") & real26.llave_completa].copy()
real_semanal = real26.groupby(K + ["semana_inicio"], as_index=False).agg(
    y_nuevo=("tallos", "sum"), ultimo_dia_real=("fecha_corte", "max"))
ultima_fecha = real26.fecha_corte.max()
ultima_semana = ultima_fecha - pd.Timedelta(days=ultima_fecha.weekday())
ultima_semana_completa = ultima_semana if ultima_fecha.weekday() == 6 else ultima_semana - pd.Timedelta(days=7)

casos = base.loc[base.finca.eq("GAITANA")].copy()
casos = casos.merge(real_semanal, left_on=K + ["semana_objetivo"],
                    right_on=K + ["semana_inicio"], how="left", validate="many_to_one")
casos["y_eval"] = casos.y_nuevo.where(casos.y_nuevo.notna(), casos.y)
casos["admisible_actual"] = casos.media_4.notna() & casos.observaciones_previas.ge(26)
casos["horizonte"] = casos.horizonte.astype(int)
model_data = casos.loc[casos.y_eval.notna() & casos.admisible_actual].copy()

# Mismo cruce nominal estricto del benchmark oficial: suma las líneas de la misma
# serie/origen/objetivo/horizonte y excluye cantidades incompletas y envíos posteriores.
e = pd.read_parquet(DATA / "estimados_semanales_limpio.parquet")
e = e.loc[e.finca.eq("GAITANA") & e.en_h1_h5 & ~e.es_ajustado
          & e.origen_solicitado.dt.year.eq(2026) & e.semana_inicio.dt.year.eq(2026)].copy()
e = e.rename(columns={"origen_solicitado": "origen", "semana_inicio": "semana_objetivo"})
engineer = e.groupby(CASE, dropna=False).agg(
    ingeniero=("estimado", "sum"), lineas=("estimado", "size"),
    cantidades=("estimado", "count"), archivos=("archivo_origen", "nunique"),
    posterior=("archivo_es_posterior", "max")).reset_index()
engineer["ingeniero_completo"] = engineer.lineas.eq(engineer.cantidades) & engineer.archivos.eq(1) & ~engineer.posterior.fillna(False)
casos = casos.merge(engineer, on=CASE, how="inner", validate="one_to_one")
casos = casos.loc[casos.ingeniero_completo & casos.y_eval.notna() & casos.admisible_actual].copy()
casos = casos.loc[casos.semana_objetivo.gt(pd.Timestamp("2026-08-31"))
                  & casos.semana_objetivo.le(ultima_semana_completa)].copy()
print("Última fecha real:", ultima_fecha.date(), "| última semana completa:", ultima_semana_completa.date())
print("Casos nuevos comparables tras cruce estricto:", len(casos))
print("Semanas objetivo:", sorted(casos.semana_objetivo.dt.strftime("%Y-%m-%d").unique()))
print("Clima disponible hasta:", pd.to_datetime(pd.read_parquet(DATA / "clima_limpio.parquet").fecha).max().date())


Última fecha real: 2026-10-03 | última semana completa: 2026-09-21
Casos nuevos comparables tras cruce estricto: 1226
Semanas objetivo: ['2026-09-07', '2026-09-14', '2026-09-21']
Clima disponible hasta: 2026-09-21


## 2. Definir el modelo sin usar el estimado del ingeniero

El modelo pronostica el ajuste sobre la media de produccion de las cuatro semanas anteriores. Cuando hay teoria aplicable y suficiente historial, aprende por separado el ajuste sobre esa teoria. Se conserva la regla de cobertura de clima del cuaderno 24: se usan sus variables solo cuando la cobertura basta; si no, el modelo funciona sin clima.

Se comparan tres referencias ya calculadas: modelo agrupado, modelos separados por horizonte y combinacion H1 agrupado + H2-H5 separados. El nuevo candidato sustituye las predicciones de MINICARNATION y CARNATION en H3-H5 con modelos entrenados para cada producto y horizonte. No toma como entrada la prediccion del ingeniero.


In [2]:
CATS = ["producto", "color", "variedad"]
MEMORY = ["produccion_lag1", "produccion_lag2", "produccion_lag3", "media_4", "media_8",
          "desv_4", "tendencia_4_8", "cambio_reciente", "produccion_52"]
CLIMATE = ["temperatura_semana_lag1", "temperatura_semana_media4",
           "humedad_semana_lag1", "humedad_semana_media4",
           "radiacion_semana_lag1", "radiacion_semana_media4"]
THEORY = ["teoria_modelo", "antiguedad_curva", "desviacion_reciente_teoria",
          "teoria_previa_1", "teoria_previa_2", "revision_teoria"]
BASE_X = CATS + ["horizonte", "semana_seno", "semana_coseno"] + MEMORY

def metricas(g, p):
    y = np.asarray(g.y_eval, dtype=float); p = np.asarray(p, dtype=float)
    den = y.sum()
    return {"n": len(y), "MAE_tallos": float(np.mean(np.abs(y-p))),
            "WAPE_pct": float(100*np.abs(y-p).sum()/den) if den else np.nan,
            "sesgo_pct": float(100*(p-y).sum()/den) if den else np.nan}

def elegir_clima(train, test, habilitar=True, minimo=0.80):
    """Usa clima sólo si existe cobertura suficiente en el ajuste evaluado."""
    if not habilitar or not all(c in train and c in test for c in CLIMATE):
        return [], 0.0, "sin clima: columnas no disponibles"
    cov_train = float(train[CLIMATE].notna().mean().mean())
    latest_origin = test.origen.max()
    current = test.loc[test.origen.eq(latest_origin), CLIMATE]
    cov_current = float(current.notna().mean().mean()) if len(current) else 0.0
    cobertura = min(cov_train, cov_current)
    if cobertura < minimo:
        return [], cobertura, f"sin clima: cobertura {cobertura:.1%} < {minimo:.0%}"
    return CLIMATE, cobertura, f"con clima: cobertura {cobertura:.1%}"

def crear_modelo(cols, params=None):
    params = params or {}
    cats = [c for c in CATS if c in cols]
    nums = [c for c in cols if c not in cats]
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), cats),
        ("num", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), nums),
    ])
    reg = HistGradientBoostingRegressor(
        learning_rate=params.get("learning_rate", .08), max_iter=params.get("max_iter", 80),
        max_leaf_nodes=params.get("max_leaf_nodes", 15), min_samples_leaf=40,
        l2_regularization=1.0, early_stopping=False, random_state=42)
    return make_pipeline(prep, reg)

def pronosticar_ruta(train, test, cols, params=None):
    """Ajusta la media4 y, con soporte suficiente, la teoría completa por separado."""
    train = train.loc[train.y_eval.notna() & train.media_4.notna()].copy()
    pred = pd.Series(index=test.index, dtype=float)
    if train.empty or test.empty: return pred
    if len(train) > 5000: train = train.sample(5000, random_state=42)
    with threadpool_limits(limits=2):
        fallback = crear_modelo(cols, params)
        fallback.fit(train[cols], train.y_eval-train.media_4)
        pred.loc[test.index] = np.maximum(0, test.media_4 + fallback.predict(test[cols]))
        tr = train.loc[train.curva_completa.fillna(False) & train.teoria_modelo.notna()]
        te = test.loc[test.curva_completa.fillna(False) & test.teoria_modelo.notna()]
        if len(tr) >= 40 and len(te):
            use = [c for c in cols + THEORY if c in train.columns]
            teoria = crear_modelo(use, params)
            teoria.fit(tr[use], tr.y_eval-tr.teoria_modelo)
            pred.loc[te.index] = np.maximum(0, te.teoria_modelo + teoria.predict(te[use]))
    return pred

def fuera_producto(df, patron):
    return df.producto.isin(str(patron).split("|"))

def evaluar_origenes(data, grupos="agrupado", excluir_entrenamiento=None, clima="auto", params=None, train_pool=None):
    """Reentrena sólo con semanas reales cerradas antes de cada origen evaluado."""
    train_pool = model_data if train_pool is None else train_pool
    salidas=[]; controles=[]
    for origen, test in data.groupby("origen", sort=True):
        test = test.copy()
        train = train_pool.loc[train_pool.semana_objetivo + pd.Timedelta(days=7) <= origen].copy()
        if excluir_entrenamiento:
            train = train.loc[~fuera_producto(train, excluir_entrenamiento)]
        if train.empty: continue
        climat, cov, modo = elegir_clima(train, test, habilitar=(clima != "sin"))
        x = BASE_X + climat
        test["prediccion"] = np.nan
        grp = test.horizonte if grupos == "horizonte" else np.where(test.horizonte <= 2, "H1_H2", "H3_H5")
        test["_grp"] = grp
        train["_grp"] = train.horizonte if grupos == "horizonte" else np.where(train.horizonte <= 2, "H1_H2", "H3_H5")
        for gval, te in test.groupby("_grp"):
            tr = train.loc[train._grp.eq(gval)]
            if tr.empty: continue
            teorio = x + THEORY
            te["prediccion"] = pronosticar_ruta(tr, te, x, params)
            test.loc[te.index, "prediccion"] = te.prediccion
            if test.loc[te.index, "prediccion"].notna().any():
                controles.append({"origen": origen, "grupo": str(gval), "train_n": len(tr),
                                  "test_n": len(te), "clima": modo, "cobertura_clima": cov})
        test.drop(columns="_grp", inplace=True)
        salidas.append(test)
    if not salidas: return pd.DataFrame(), pd.DataFrame(controles)
    return pd.concat(salidas).dropna(subset=["prediccion"]), pd.DataFrame(controles)

def resumen_comparacion(pred):
    rows=[]
    for nombre,col in [("Modelo", "prediccion"),("Ingeniero","ingeniero"),("Media 4 semanas","media_4"),("Teoría aplicable","teoria_modelo")]:
        g=pred.loc[pred[col].notna()].copy()
        if len(g): rows.append({"metodo":nombre,**metricas(g,g[col])})
    return pd.DataFrame(rows)


## 3. Recuperar las predicciones de referencia y congelar el tuning

Los parametros se eligen usando el tuning de noviembre guardado por el cuaderno 24. Las predicciones agrupadas y separadas provienen de ese mismo experimento y corresponden a las mismas filas comparables.


In [3]:
# These saved predictions were generated using information available at each forecast origin.
OUT = DATA / "experimentos_actualizacion_20261020"
keys = CASE
group_pred = pd.read_parquet(OUT / "24_predicciones_agrupado.parquet")
separate_pred = pd.read_parquet(OUT / "24_predicciones_horizontes.parquet")
for frame in [group_pred, separate_pred]:
    frame["semana_objetivo"] = pd.to_datetime(frame.semana_objetivo)
feature_cols = [x for x in BASE_X + THEORY + ["curva_completa"] if x in group_pred.columns and x not in keys]
base = group_pred[keys + feature_cols + ["y_eval", "ingeniero", "prediccion"]].rename(columns={"prediccion":"pred_agrupado"}).merge(
    separate_pred[keys + ["prediccion"]].rename(columns={"prediccion":"pred_separado"}),
    on=keys, how="inner", validate="one_to_one")
base["pred_hibrida"] = base.pred_separado.where(base.horizonte.ne(1), base.pred_agrupado)
# Product/horizon tuning is inherited from November; no recent test labels choose tree settings.
tuning = pd.read_csv(OUT / "24_tuning_noviembre.csv")
params_h = {}
for h in range(1,6):
    t = tuning.loc[tuning.familia.eq(f"Modelo H{h}")].sort_values("WAPE_pct")
    if t.empty: raise ValueError(f"No tuning row available for H{h}")
    params_h[h] = {"max_iter":int(t.iloc[0].max_iter), "max_leaf_nodes":int(t.iloc[0].max_leaf_nodes)}
print("Parametros heredados del tuning previo:", params_h)
print("Casos comunes con predicciones de referencia:", len(base))



Parametros heredados del tuning previo: {1: {'max_iter': 80, 'max_leaf_nodes': 15}, 2: {'max_iter': 80, 'max_leaf_nodes': 15}, 3: {'max_iter': 40, 'max_leaf_nodes': 15}, 4: {'max_iter': 40, 'max_leaf_nodes': 15}, 5: {'max_iter': 80, 'max_leaf_nodes': 15}}
Casos comunes con predicciones de referencia: 1226


## 4. Entrenar los modelos especializados por producto y horizonte

En cada origen se filtra el historial para que la produccion objetivo ya estuviera cerrada. Para cada uno de los dos productos priorizados y cada horizonte H3-H5, se ajusta un modelo propio. Sus parametros se seleccionan con origenes de noviembre de 2025 y entrenamiento cronologico previo a cada origen. Se requiere un minimo de 80 filas de entrenamiento; si un segmento no alcanza ese soporte, se conserva la prediccion separada por horizonte. Para productos distintos de los dos priorizados tambien se conserva esa prediccion.

Despues se probara un ensamble entre dos pronosticos propios (el separado por horizonte y el especializado); no entra el pronostico del ingeniero. El candidato completo es por tanto un sistema de pronostico propio: H1 agrupado, H2-H5 separados y modelos de producto-horizonte para MINICARNATION y CARNATION en H3-H5.


In [4]:
TOP_PRODUCTS = ["MINICARNATION", "CARNATION"]
MIN_TRAIN = 80
# Tune only on older November origins. The recent evaluation window stays untouched.
product_grid = [{"max_iter":40,"max_leaf_nodes":15},{"max_iter":80,"max_leaf_nodes":15}]
valid_nov_product = model_data.loc[
    model_data.origen.between("2025-11-03","2025-11-24")
    & model_data.producto.isin(TOP_PRODUCTS) & model_data.horizonte.isin([3,4,5])
].copy()
params_product = {}
product_tuning_rows=[]
for product in TOP_PRODUCTS:
    for h in [3,4,5]:
        validation = valid_nov_product.loc[valid_nov_product.producto.eq(product) & valid_nov_product.horizonte.eq(h)]
        for setting in product_grid:
            pieces=[]
            for origin, te_nov in validation.groupby("origen",sort=True):
                tr_nov = model_data.loc[model_data.producto.eq(product) & model_data.horizonte.eq(h)
                    & model_data.y_eval.notna() & model_data.media_4.notna()
                    & (model_data.semana_objetivo + pd.Timedelta(days=7) <= origin)].copy()
                if len(tr_nov)<MIN_TRAIN: continue
                climate_nov,_,_=elegir_clima(tr_nov,te_nov,habilitar=True)
                pred_nov=pronosticar_ruta(tr_nov,te_nov,BASE_X+climate_nov,setting)
                tmp=te_nov.loc[pred_nov.notna(),["y_eval"]].copy(); tmp["pred"]=pred_nov.dropna()
                pieces.append(tmp)
            if pieces:
                joined=pd.concat(pieces); den=joined.y_eval.sum()
                score=100*(joined.pred-joined.y_eval).abs().sum()/den if den else np.nan
                product_tuning_rows.append({"producto":product,"horizonte":h,**setting,"n_validacion":len(joined),"WAPE_noviembre_pct":score})
product_tuning=pd.DataFrame(product_tuning_rows)
for product in TOP_PRODUCTS:
    for h in [3,4,5]:
        options=product_tuning.loc[product_tuning.producto.eq(product) & product_tuning.horizonte.eq(h)].sort_values("WAPE_noviembre_pct")
        if options.empty: raise ValueError(f"No temporal tuning results for {product}, H{h}")
        params_product[(product,h)]={"max_iter":int(options.iloc[0].max_iter),"max_leaf_nodes":int(options.iloc[0].max_leaf_nodes)}
print("Tuning temporal por producto-horizonte (noviembre):"); display(product_tuning.sort_values(["producto","horizonte","WAPE_noviembre_pct"]))
print("Parametros congelados para la evaluacion reciente:",params_product)
# Learn a model-only ensemble weight on November. Alpha is the weight on the product-specific model.
blend_validation_parts=[]
for product in TOP_PRODUCTS:
    for h in [3,4,5]:
        validation=valid_nov_product.loc[valid_nov_product.producto.eq(product)&valid_nov_product.horizonte.eq(h)]
        for origin,te_nov in validation.groupby("origen",sort=True):
            tr_all=model_data.loc[model_data.horizonte.eq(h)&model_data.y_eval.notna()&model_data.media_4.notna()
                &(model_data.semana_objetivo+pd.Timedelta(days=7)<=origin)].copy()
            tr_special=tr_all.loc[tr_all.producto.eq(product)].copy()
            if len(tr_special)<MIN_TRAIN: continue
            climate_all,_,_=elegir_clima(tr_all,te_nov,habilitar=True)
            pred_all=pronosticar_ruta(tr_all,te_nov,BASE_X+climate_all,params_h[h])
            climate_special,_,_=elegir_clima(tr_special,te_nov,habilitar=True)
            pred_special=pronosticar_ruta(tr_special,te_nov,BASE_X+climate_special,params_product[(product,h)])
            tmp=te_nov.loc[pred_all.notna()&pred_special.notna(),keys+["y_eval"]].copy()
            tmp["pred_general"]=pred_all.loc[tmp.index]
            tmp["pred_special"]=pred_special.loc[tmp.index]
            tmp["producto"]=product; tmp["horizonte"]=h
            blend_validation_parts.append(tmp)
blend_validation=pd.concat(blend_validation_parts,ignore_index=True)
blend_tuning_rows=[]; blend_weights={}
for product in TOP_PRODUCTS:
    for h in [3,4,5]:
        v=blend_validation.loc[blend_validation.producto.eq(product)&blend_validation.horizonte.eq(h)]
        trials=[]
        for alpha in [0,.25,.5,.75,1]:
            pred=(1-alpha)*v.pred_general+alpha*v.pred_special
            den=v.y_eval.sum(); score=100*(pred-v.y_eval).abs().sum()/den if den else np.nan
            trials.append((score,alpha,len(v)))
            blend_tuning_rows.append({"producto":product,"horizonte":h,"peso_modelo_especializado":alpha,"n_validacion":len(v),"WAPE_noviembre_pct":score})
        if not trials: raise ValueError(f"No blend validation rows for {product}, H{h}")
        blend_weights[(product,h)]=min(trials)[1]
blend_tuning=pd.DataFrame(blend_tuning_rows)
print("Peso del modelo especializado elegido solo con noviembre:"); display(blend_tuning.sort_values(["producto","horizonte","WAPE_noviembre_pct"]))
base["pred_especializado"] = base.pred_hibrida
base["pred_candidato"] = base.pred_hibrida
train_log=[]
for origin, test_origin in casos.groupby("origen", sort=True):
    train_origin = model_data.loc[
        model_data.y_eval.notna() & model_data.media_4.notna()
        & (model_data.semana_objetivo + pd.Timedelta(days=7) <= origin)
    ].copy()
    for h in [3,4,5]:
        for product in TOP_PRODUCTS:
            test_mask = base.origen.eq(origin) & base.horizonte.eq(h) & base.producto.eq(product)
            te = base.loc[test_mask].copy()
            if te.empty: continue
            tr = train_origin.loc[train_origin.horizonte.eq(h) & train_origin.producto.eq(product)].copy()
            if len(tr) < MIN_TRAIN:
                train_log.append({"origen":origin,"horizonte":h,"producto":product,"n_train":len(tr),"n_test":len(te),"estado":"soporte bajo; conserva modelo por horizonte"})
                continue
            climate, coverage, climate_note = elegir_clima(train_origin, te, habilitar=True)
            cols = BASE_X + climate
            # The engineer estimate is intentionally absent from cols and training labels.
            if "ingeniero" in cols or "ingeniero" in tr.columns and "ingeniero" in cols:
                raise AssertionError("Engineer estimate must not be a model feature")
            fitted = pronosticar_ruta(tr, te, cols, params_product[(product,h)])
            base.loc[te.index,"pred_especializado"] = fitted
            train_log.append({"origen":origin,"horizonte":h,"producto":product,"n_train":len(tr),
                              "n_test":len(te),"clima":climate_note,"estado":"modelo especializado"})
train_log=pd.DataFrame(train_log)
base["peso_modelo_especializado"]=0.0
for (product,h),alpha in blend_weights.items():
    mask=base.producto.eq(product)&base.horizonte.eq(h)&base.horizonte.ge(3)
    base.loc[mask,"peso_modelo_especializado"]=alpha
    base.loc[mask,"pred_candidato"]=(1-alpha)*base.loc[mask,"pred_separado"]+alpha*base.loc[mask,"pred_especializado"]
print("Resumen del entrenamiento por origen-producto-horizonte:")
display(train_log)
print("Casos candidatos que conservaron el modelo por horizonte:", int(base.pred_candidato.eq(base.pred_hibrida).sum()))


Tuning temporal por producto-horizonte (noviembre):


,producto,horizonte,max_iter,max_leaf_nodes,n_validacion,WAPE_noviembre_pct
7,CARNATION,3,80,15,195,35.232302
6,CARNATION,3,40,15,195,36.775259
9,CARNATION,4,80,15,189,39.992128
8,CARNATION,4,40,15,189,40.871929
11,CARNATION,5,80,15,185,39.751163
10,CARNATION,5,40,15,185,40.243105
0,MINICARNATION,3,40,15,137,21.973097
1,MINICARNATION,3,80,15,137,22.140052
2,MINICARNATION,4,40,15,134,22.895712
3,MINICARNATION,4,80,15,134,23.240781


Parametros congelados para la evaluacion reciente: {('MINICARNATION', 3): {'max_iter': 40, 'max_leaf_nodes': 15}, ('MINICARNATION', 4): {'max_iter': 40, 'max_leaf_nodes': 15}, ('MINICARNATION', 5): {'max_iter': 40, 'max_leaf_nodes': 15}, ('CARNATION', 3): {'max_iter': 80, 'max_leaf_nodes': 15}, ('CARNATION', 4): {'max_iter': 80, 'max_leaf_nodes': 15}, ('CARNATION', 5): {'max_iter': 80, 'max_leaf_nodes': 15}}


Peso del modelo especializado elegido solo con noviembre:


,producto,horizonte,peso_modelo_especializado,n_validacion,WAPE_noviembre_pct
19,CARNATION,3,1.00,195,35.232302
18,CARNATION,3,0.75,195,36.454092
17,CARNATION,3,0.50,195,38.045167
16,CARNATION,3,0.25,195,39.821458
15,CARNATION,3,0.00,195,41.840368
24,CARNATION,4,1.00,189,39.992128
23,CARNATION,4,0.75,189,41.217955
22,CARNATION,4,0.50,189,42.650426
21,CARNATION,4,0.25,189,44.521854
20,CARNATION,4,0.00,189,46.759465


Resumen del entrenamiento por origen-producto-horizonte:


,origen,horizonte,producto,n_train,n_test,clima,estado
0,2026-08-10,5,MINICARNATION,8569,30,sin clima: columnas no disponibles,modelo especializado
1,2026-08-10,5,CARNATION,11442,39,sin clima: columnas no disponibles,modelo especializado
2,2026-08-17,4,MINICARNATION,8683,30,sin clima: columnas no disponibles,modelo especializado
3,2026-08-17,4,CARNATION,11629,39,sin clima: columnas no disponibles,modelo especializado
4,2026-08-17,5,MINICARNATION,8601,30,sin clima: columnas no disponibles,modelo especializado
5,2026-08-17,5,CARNATION,11486,38,sin clima: columnas no disponibles,modelo especializado
6,2026-08-24,3,MINICARNATION,8799,30,sin clima: columnas no disponibles,modelo especializado
7,2026-08-24,3,CARNATION,11824,39,sin clima: columnas no disponibles,modelo especializado
8,2026-08-24,4,MINICARNATION,8715,30,sin clima: columnas no disponibles,modelo especializado
9,2026-08-24,4,CARNATION,11673,38,sin clima: columnas no disponibles,modelo especializado


Casos candidatos que conservaron el modelo por horizonte: 611


## 5. Comparar el candidato con el ingeniero

La comparacion usa WAPE (menor es mejor) y sesgo porcentual (positivo: sobreestima; negativo: subestima). Se muestran el total y cada horizonte. El ingeniero es aqui un benchmark de evaluacion, nunca una entrada del candidato.


In [5]:
def score(data, col):
    y=data.y_eval.sum()
    return {"casos":len(data),"WAPE_pct":100*(data[col]-data.y_eval).abs().sum()/y,
            "sesgo_pct":100*(data[col].sum()-y)/y}
methods=[("Agrupado","pred_agrupado"),("Separado por horizonte","pred_separado"),
         ("H1 agrupado + H2-H5 separados","pred_hibrida"),("Especializado puro (solo modelo)","pred_especializado"),("Ensamble de modelos propios","pred_candidato"),("Ingeniero (referencia)","ingeniero")]
summary=pd.DataFrame([{"estrategia":name,**score(base,col)} for name,col in methods])
rows=[]
for h,g in base.groupby("horizonte"):
    for name,col in methods: rows.append({"horizonte":h,"estrategia":name,**score(g,col)})
detail=pd.DataFrame(rows)
print("Resultado total (misma poblacion):")
display(summary.round(2))
print("Resultado por horizonte:")
display(detail.round(2))
print("Foco H3-H5:")
display(detail.loc[detail.horizonte.ge(3)].round(2))


Resultado total (misma poblacion):


,estrategia,casos,WAPE_pct,sesgo_pct
0,Agrupado,1226,18.26,-5.48
1,Separado por horizonte,1226,17.86,-6.17
2,H1 agrupado + H2-H5 separados,1226,17.80,-6.26
3,Especializado puro (solo modelo),1226,17.59,-4.12
4,Ensamble de modelos propios,1226,17.53,-4.22
5,Ingeniero (referencia),1226,16.34,-7.51


Resultado por horizonte:


,horizonte,estrategia,casos,WAPE_pct,sesgo_pct
0,1,Agrupado,103,12.28,2.75
1,1,Separado por horizonte,103,12.94,3.75
2,1,H1 agrupado + H2-H5 separados,103,12.28,2.75
3,1,Especializado puro (solo modelo),103,12.28,2.75
4,1,Ensamble de modelos propios,103,12.28,2.75
5,1,Ingeniero (referencia),103,15.23,-7.79
6,2,Agrupado,205,15.06,1.97
7,2,Separado por horizonte,205,14.96,-0.02
8,2,H1 agrupado + H2-H5 separados,205,14.96,-0.02
9,2,Especializado puro (solo modelo),205,14.96,-0.02


Foco H3-H5:


,horizonte,estrategia,casos,WAPE_pct,sesgo_pct
12,3,Agrupado,306,17.02,-9.01
13,3,Separado por horizonte,306,16.77,-9.73
14,3,H1 agrupado + H2-H5 separados,306,16.77,-9.73
15,3,Especializado puro (solo modelo),306,17.27,-5.85
16,3,Ensamble de modelos propios,306,17.27,-5.85
17,3,Ingeniero (referencia),306,16.51,-7.99
18,4,Agrupado,306,19.65,-6.68
19,4,Separado por horizonte,306,19.47,-7.04
20,4,H1 agrupado + H2-H5 separados,306,19.47,-7.04
21,4,Especializado puro (solo modelo),306,18.89,-3.96


## 6. Ver donde mejora y donde sigue la brecha

El corte por producto permite verificar si especializar MINICARNATION y CARNATION ayudo o si solo movio el error. Se muestran tambien teoria disponible y tendencia reciente para orientar la siguiente hipotesis. Estos cortes son diagnosticos, no pruebas de causalidad.


In [6]:
focus=base.loc[base.horizonte.ge(3)].copy()
focus["err_candidato"]=(focus.pred_candidato-focus.y_eval).abs()
focus["err_ingeniero"]=(focus.ingeniero-focus.y_eval).abs()
focus["err_firmado_candidato"]=focus.pred_candidato-focus.y_eval
def group_scores(data,keys):
    d=data.groupby(keys,dropna=False).agg(casos=("y_eval","size"),real=("y_eval","sum"),
        error_candidato=("err_candidato","sum"),error_ingeniero=("err_ingeniero","sum"),
        bias_tallos=("err_firmado_candidato","sum")).reset_index()
    d["WAPE_candidato_pct"]=100*d.error_candidato/d.real.replace(0,np.nan)
    d["WAPE_ingeniero_pct"]=100*d.error_ingeniero/d.real.replace(0,np.nan)
    d["sesgo_candidato_pct"]=100*d.bias_tallos/d.real.replace(0,np.nan)
    d["brecha_error_tallos"]=d.error_candidato-d.error_ingeniero
    return d.sort_values("brecha_error_tallos",ascending=False)
print("Por producto y horizonte:")
display(group_scores(focus,["horizonte","producto"]).head(20).round(2))
if "teoria_modelo" in focus:
    focus["estado_teoria"]=np.where(focus.teoria_modelo.notna(),"Teoria disponible","Sin teoria utilizable")
    print("Por teoria disponible:"); display(group_scores(focus,["horizonte","estado_teoria"]).round(2))
if "tendencia_4_8" in focus:
    focus["estado_tendencia"]=np.select([focus.tendencia_4_8.gt(0),focus.tendencia_4_8.le(0)],
        ["Tendencia positiva","Tendencia estable o negativa"],default="Sin tendencia")
    print("Por tendencia reciente:"); display(group_scores(focus,["horizonte","estado_tendencia"]).round(2))


Por producto y horizonte:


,horizonte,producto,casos,real,error_candidato,error_ingeniero,bias_tallos,WAPE_candidato_pct,WAPE_ingeniero_pct,sesgo_candidato_pct,brecha_error_tallos
24,5,CARNATION,115,821022,253792.41,177518.0,-192666.92,30.91,21.62,-23.47,76274.41
26,5,MINICARNATION,90,2307566,359317.95,295146.0,-21582.1,15.57,12.79,-0.94,64171.95
15,4,MINICARNATION,90,2307566,363966.36,299856.0,63481.06,15.77,12.99,2.75,64110.36
13,4,CARNATION,115,821022,221957.2,178198.0,-166317.3,27.03,21.7,-20.26,43759.2
2,3,CARNATION,115,821022,210719.32,178838.0,-176158.54,25.67,21.78,-21.46,31881.32
4,3,MINICARNATION,90,2307566,320596.99,300234.0,-3162.25,13.89,13.01,-0.14,20362.99
18,4,STAR,3,24445,10880.06,5205.0,10880.06,44.51,21.29,44.51,5675.06
7,3,STAR,3,24445,10822.27,5205.0,10822.27,44.27,21.29,44.27,5617.27
23,5,BARBATUS,15,23760,8858.3,4520.0,501.44,37.28,19.02,2.11,4338.3
29,5,STAR,3,24445,9166.81,5295.0,9166.81,37.5,21.66,37.5,3871.81


Por teoria disponible:


,horizonte,estado_teoria,casos,real,error_candidato,error_ingeniero,bias_tallos,WAPE_candidato_pct,WAPE_ingeniero_pct,sesgo_candidato_pct,brecha_error_tallos
4,5,Sin teoria utilizable,283,3780183,738362.22,637101.0,-288055.11,19.53,16.85,-7.62,101261.22
2,4,Sin teoria utilizable,283,3754623,694172.4,625863.0,-133763.16,18.49,16.67,-3.56,68309.4
0,3,Sin teoria utilizable,282,3753589,643013.07,613149.0,-219472.31,17.13,16.34,-5.85,29864.07
3,4,Teoria disponible,23,170048,47283.48,33276.0,-21469.37,27.81,19.57,-12.63,14007.48
5,5,Teoria disponible,23,144488,37226.82,28206.0,-16648.46,25.76,19.52,-11.52,9020.82
1,3,Teoria disponible,24,171082,34944.89,34968.0,-10252.96,20.43,20.44,-5.99,-23.11


Por tendencia reciente:


,horizonte,estado_tendencia,casos,real,error_candidato,error_ingeniero,bias_tallos,WAPE_candidato_pct,WAPE_ingeniero_pct,sesgo_candidato_pct,brecha_error_tallos
8,5,Tendencia positiva,132,1750719,362836.83,294755.0,-179981.65,20.73,16.84,-10.28,68081.83
5,4,Tendencia positiva,144,1787092,380365.85,313056.0,-110304.08,21.28,17.52,-6.17,67309.85
7,5,Tendencia estable o negativa,171,2165912,409920.16,367852.0,-121889.88,18.93,16.98,-5.63,42068.16
1,3,Tendencia estable o negativa,137,1776722,303334.89,273870.0,-60712.51,17.07,15.41,-3.42,29464.89
4,4,Tendencia estable o negativa,160,2131639,358403.98,343703.0,-42242.4,16.81,16.12,-1.98,14700.98
0,3,Sin tendencia,1,3420,2197.42,1660.0,-2197.42,64.25,48.54,-64.25,537.42
3,4,Sin tendencia,2,5940,2686.05,2380.0,-2686.05,45.22,40.07,-45.22,306.05
6,5,Sin tendencia,3,8040,2832.04,2700.0,-2832.04,35.22,33.58,-35.22,132.04
2,3,Tendencia positiva,168,2144529,372425.66,372587.0,-166815.35,17.37,17.37,-7.78,-161.34


## 7. Conclusion y decision

El candidato no usa el estimado del ingeniero para pronosticar. Si el WAPE baja frente al ingeniero en H3-H5 y mantiene un sesgo aceptable, es una alternativa propia que merece una nueva prueba temporal. Si no lo hace, el resultado indica que la especializacion por esos dos productos no basta; se conserva el mejor modelo actual como referencia de investigacion.

La ventana reciente tiene pocas semanas objetivo. El modelo especializado se entreno con historia anterior a cada origen, pero el test observado sigue siendo corto. Para afirmar reemplazo estable, repetir la evaluacion en varias semanas futuras cerradas y comparar exactamente la misma poblacion. No se debe recalibrar usando las semanas de test.


In [7]:
# Save forecasts and tables for review.
blend_tuning.to_csv(OUT/"25_tuning_ensamble_modelos_noviembre.csv",index=False)
product_tuning.to_csv(OUT/"25_tuning_producto_horizonte_noviembre.csv",index=False)
summary.to_csv(OUT/"25_resumen_modelos_sin_ingeniero.csv",index=False)
detail.to_csv(OUT/"25_detalle_modelos_sin_ingeniero_por_horizonte.csv",index=False)
train_log.to_csv(OUT/"25_entrenamiento_modelos_especializados.csv",index=False)
base.to_parquet(OUT/"25_predicciones_candidato_sin_ingeniero.parquet",index=False)
print("Resultados guardados en",OUT)


Resultados guardados en

 C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto\experimentos_actualizacion_20261020
